# NB1 – ES Bán hàng: khảo sát, tạo nhãn và đặc trưng doanh số, EDA tín hiệu thời gian
**DS317 – Lab 3 | Nhóm 5 (DS317.R11)** · Đề tài: dự báo doanh số và cảnh báo rủi ro tồn kho theo SKU–tuần (Electronics Sales)

| Phần | Nội dung | Yêu cầu task |
|---|---|---|
| 0 | Cấu hình chung + nạp dataset từ file zip | (1) |
| 1 | Đọc 27 bảng `es_<tên>_raw`, danh mục bảng, thời gian phủ, chất lượng 6 chiều, ngoại lai, riêng tư (`CustomerName`), Hình 1 | (2) |
| 2 | Lọc đơn hợp lệ → nối → tổng hợp SKU–tuần → đặc trưng/nhãn `y_qty`, thống kê phân bố, Hình 2, Hình 3 | (2), (3) |
| 3 | EDA tín hiệu thời gian: xu hướng, mùa vụ, autocorrelation, quan hệ tồn kho–tốc độ bán, kết luận | (4) |
| 4 | Kiểm tra khóa → ghi file xuất | (1) |

**File xuất vào `interim/`:** `es_sales_wk`, `es_filter_log`, `es_quality_sales`, `es_table_catalog`
**Cách chạy (Colab):** Runtime → Run all → khi hộp thoại hiện ra, chọn file **`archive.zip`** (dataset Electronics Sales). Nếu zip đã nằm trong `/content` hoặc Drive thì notebook tự tìm thấy.

## 0. Cấu hình chung và nạp dataset

In [ ]:
# ============================ Ô CẤU HÌNH CHUNG (giữ GIỐNG HỆT ở mọi notebook của nhóm) ============================
import os, sys, re, json, math, time, random, zipfile, shutil, warnings, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.dates as mdates
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 90)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ---------------- Hằng số (khai báo MỘT LẦN) ----------------
SEED              = 42
HORIZON_WEEKS     = 4                 # nhìn trước 4 tuần
EPS               = 1e-6              # chặn mẫu số của Coverage
SYNC_START        = '2023-07-03'      # A2: bán hàng và tồn kho cùng bắt đầu 01/07/2023 (thứ Bảy) -> tuần ĐỦ 7 ngày đầu tiên
SYNC_END          = '2025-12-22'      # A2: dữ liệu kết thúc 31/12/2025 (thứ Tư) -> tuần ĐỦ 7 ngày cuối cùng (tuần 29/12 chỉ có 3 ngày, bị loại)
MIN_HISTORY_WEEKS = 26                # SKU phải có >= N tuần trong TRAIN (áp dụng ở notebook chung)
REF_DATE          = None              # ngày lớn nhất trong dữ liệu -> được tính ngay sau khi đọc dữ liệu (xem set_ref_date)
KEY               = ['sku_id', 'week_start']
ACCESS_DATE_ES    = '🟦 dd/mm/yyyy'   # A1: ngày truy cập Kaggle (điền cho khớp Mục 1 của notebook chung)

# ---------------- Đường dẫn ----------------
USE_DRIVE = True       # True: lưu raw/interim trên Google Drive (để NB1 -> NB2 -> notebook chung dùng chung file). False: lưu tạm trong /content
IN_COLAB  = 'google.colab' in sys.modules
BASE = Path('.')
if IN_COLAB:
    BASE = Path('/content')
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive'); BASE = Path('/content/drive/MyDrive')
        except Exception as e:
            print('⚠️ Không mount được Drive, dùng /content (file sẽ mất khi hết phiên):', e)
ROOT = BASE / 'DS317_Nhom5'
DIRS = {k: ROOT / k for k in ['raw', 'interim', 'processed', 'results', 'figures', 'excel']}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

RAW_TABLES = ['BrandSubcategoryMap', 'Brands', 'Categories', 'CustomerReturns', 'Customers', 'Governorates',
              'InventoryBalances', 'InventoryMovements', 'Locations', 'PaymentMethods', 'Payments', 'ProductVariants',
              'Products', 'Promotions', 'PurchaseOrderLines', 'PurchaseOrders', 'PurchaseReceiptLines', 'PurchaseReceipts',
              'PurchaseReturns', 'SKUSuppliers', 'SalesChannels', 'SalesOrderLines', 'SalesOrders', 'Shipments',
              'ShippingProviders', 'Subcategories', 'Suppliers']          # 27 bảng của Electronics Sales

# ---------------- Hàm dùng chung ----------------
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
set_seed()

def to_snake(name):                                   # SalesOrderLines -> sales_order_lines ; SKUSuppliers -> sku_suppliers
    return re.sub(r'(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])', '_', name).lower()

def week_start_of(s):                                 # ĐỊNH NGHĨA TUẦN DUY NHẤT: thứ Hai 00:00
    s = pd.to_datetime(s)
    return (s - pd.to_timedelta(s.dt.weekday, unit='D')).dt.normalize().astype('datetime64[ns]')

def is_valid_order(df, status_col='OrderStatus', bad=('Cancelled', 'Canceled', 'Failed')):
    # A2: OrderStatus gồm Completed/Complete (hợp lệ) và Cancelled/Canceled (hủy) -> loại các đơn hủy
    return ~df[status_col].isin(bad)

def robust_range(dates, min_share=0.05):
    """Khoảng ngày 'dày đặc': bỏ các ngày lẻ tẻ (số bản ghi < 5% trung vị số bản ghi/ngày) coi như ngoại lai thời gian."""
    d = pd.to_datetime(dates, errors='coerce').dropna().dt.normalize()
    if d.empty: return pd.NaT, pd.NaT
    c = d.value_counts(); keep = c[c >= max(1, min_share * c.median())].index
    return keep.min(), keep.max()

def set_ref_date(movements_raw):                      # REF_DATE = ngày lớn nhất (dày đặc) của nhật ký tồn kho
    global REF_DATE
    REF_DATE = robust_range(movements_raw['MovementDateTime'])[1]
    return REF_DATE

def read_tables(names, verbose=True):                 # đọc bảng thành es_<tên_bảng_snake>_raw
    out = {}
    for t in names:
        df = pd.read_csv(DIRS['raw'] / f'{t}.csv', low_memory=False)
        out[t] = df; globals()[f'es_{to_snake(t)}_raw'] = df
        if verbose: print(f'  es_{to_snake(t)}_raw'.ljust(36), f'{df.shape[0]:>9,} dòng × {df.shape[1]:>2} cột')
    return out

def prepare_raw():
    """Chỉ cần đưa file zip dataset vào Colab (kéo thả vào /content, hoặc để trong Drive, hoặc chọn file khi hộp thoại hiện ra)."""
    raw = DIRS['raw']; miss = [t for t in RAW_TABLES if not (raw / f'{t}.csv').exists()]
    if not miss: print(f'✅ Đã có đủ {len(RAW_TABLES)} bảng trong {raw}'); return
    def find_zip():
        cands = []
        for b in [Path('/content'), Path('.'), ROOT, raw, BASE]:
            if b.exists(): cands += sorted(b.glob('*.zip'))
        for c in cands:
            try:
                with zipfile.ZipFile(c) as z:
                    if any(n.endswith('SalesOrderLines.csv') for n in z.namelist()): return c
            except Exception: pass
        return None
    z_path = find_zip()
    if z_path is None and IN_COLAB:
        print('📦 Chưa thấy file zip dataset – hãy chọn file archive.zip (Electronics Sales) ở hộp thoại bên dưới...')
        from google.colab import files
        files.upload(); z_path = find_zip()
    if z_path is None:
        raise FileNotFoundError('Không tìm thấy file zip chứa SalesOrderLines.csv. Hãy đặt archive.zip cạnh notebook (hoặc /content, hoặc ' + str(ROOT) + ').')
    print('📦 Giải nén', z_path, '->', raw)
    with zipfile.ZipFile(z_path) as z:
        for m in z.namelist():
            if m.lower().endswith('.csv'):
                with z.open(m) as src, open(raw / Path(m).name, 'wb') as dst: shutil.copyfileobj(src, dst)
    miss = [t for t in RAW_TABLES if not (raw / f'{t}.csv').exists()]
    assert not miss, f'Thiếu bảng sau khi giải nén: {miss}'
    print(f'✅ Đã có đủ {len(RAW_TABLES)} bảng')

def save_df(df, name, folder='interim'):
    p = DIRS[folder] / name
    try:
        df.to_parquet(p.with_suffix('.parquet'), index=False)
        p.with_suffix('.csv').unlink(missing_ok=True); return p.with_suffix('.parquet')
    except Exception:
        df.to_csv(p.with_suffix('.csv'), index=False); p.with_suffix('.parquet').unlink(missing_ok=True); return p.with_suffix('.csv')

def load_df(name, folder='interim'):
    for ext in ('.parquet', '.csv'):
        p = (DIRS[folder] / name).with_suffix(ext)
        if p.exists():
            df = pd.read_parquet(p) if ext == '.parquet' else pd.read_csv(p)
            if 'week_start' in df.columns: df['week_start'] = pd.to_datetime(df['week_start'])
            return df
    return None

STEP_LOG = []
def log_step(df, step):
    STEP_LOG.append({'bước': step, 'số_dòng': len(df), 'số_SKU': df.sku_id.nunique(), 'số_tuần': df.week_start.nunique()})
    print(f'  [{step}] dòng={len(df):,} SKU={df.sku_id.nunique():,} tuần={df.week_start.nunique()}')

def save_fig(fig, name):
    p = DIRS['figures'] / f'fig_{name}.png'; fig.savefig(p, dpi=200, bbox_inches='tight'); print('đã lưu', p.name)

# Quy tắc dấu số lượng của InventoryMovements (A2, kiểm chứng khớp 100% InventoryBalances ở NB2):
#   Quantity luôn dương; tác động lên tồn KHẢ DỤNG = +Quantity nếu ToCondition=='Available', -Quantity nếu FromCondition=='Available'.
#   (Internal Transfer: từ Available -> Available nên triệt tiêu ở mức toàn chuỗi; Non-Sellable không tính vào tồn khả dụng)
def movement_delta(im):
    return im['Quantity'] * (im['ToCondition'].eq('Available').astype(int) - im['FromCondition'].eq('Available').astype(int))

def stock_end_chain_weekly(im, last_week=None):
    """Tồn kho khả dụng CUỐI TUẦN (toàn chuỗi) theo SKU-tuần = cộng dồn delta từ đầu dữ liệu. Trả về [sku_id, week_start, stock_end, delta_wk]."""
    last_week = pd.Timestamp(SYNC_END if last_week is None else last_week)
    d = pd.DataFrame({'sku_id': im['SKU_ID'].astype(str).values,
                      'week_start': week_start_of(im['MovementDateTime']).values,
                      'delta': movement_delta(im).values})
    d = d[d['week_start'] <= last_week]
    w = d.groupby(['sku_id', 'week_start'])['delta'].sum().unstack('week_start', fill_value=0)
    w = w.reindex(columns=pd.date_range(w.columns.min(), last_week, freq='7D'), fill_value=0); w.columns.name = 'week_start'
    out = w.cumsum(axis=1).stack().rename('stock_end').reset_index()
    out['delta_wk'] = w.stack().values
    return out

def check_and_save(df, name, key=None, folder='interim'):
    """Cell cuối mỗi notebook: kiểm tra khóa không trùng -> in số dòng, tuần min–max, số thiếu -> rồi mới ghi file."""
    print(f'──────── {name} ────────')
    if key is not None:
        dup = int(df.duplicated(key).sum()); assert dup == 0, f'{name}: có {dup} khóa {key} trùng'
        print(f'khóa {key} trùng: {dup}  ✅')
    print(f'số dòng: {len(df):,} | số cột: {df.shape[1]}')
    if 'week_start' in df.columns:
        df['week_start'] = pd.to_datetime(df['week_start']).astype('datetime64[ns]')
        print(f'tuần min–max: {df.week_start.min().date()} → {df.week_start.max().date()} | số tuần: {df.week_start.nunique()}')
    if 'sku_id' in df.columns: print(f'số SKU: {df.sku_id.nunique():,}')
    miss = df.isna().sum(); miss = miss[miss > 0]
    print('số giá trị thiếu theo cột:', miss.to_dict() if len(miss) else 'không có')
    assert 'CustomerName' not in df.columns, 'Không được xuất cột CustomerName (riêng tư)'
    p = save_df(df, name, folder); print('→ đã ghi', p, '\n'); return p

print('Cấu hình xong | ROOT =', ROOT.resolve(), '| SYNC:', SYNC_START, '→', SYNC_END)

In [ ]:
# ---- Nạp dataset từ file zip (chạy 1 lần; lần sau có sẵn CSV trong raw/ thì bỏ qua) ----
prepare_raw()

## 1. Khảo sát dữ liệu *(A2)*
### 1.1 Đọc 27 bảng thành `es_<tên bảng>_raw`

In [ ]:
t0 = time.time()
RAW = read_tables(RAW_TABLES)
set_ref_date(es_inventory_movements_raw)
print(f'\nĐã đọc {len(RAW)} bảng trong {time.time() - t0:.1f}s | REF_DATE (ngày lớn nhất, dày đặc, của nhật ký tồn kho) = {REF_DATE.date()}')
assert pd.Timestamp(SYNC_END) + pd.Timedelta(days=6) <= REF_DATE, 'SYNC_END vượt quá dữ liệu'

### 1.2 Danh mục bảng (dòng, cột, MB, ngày min/max) và thời gian phủ của từng bảng

In [ ]:
DATE_COL_PAT = re.compile(r'(Date|DateTime)$|^Active(From|To)$')
cov_rows, cat_rows = [], []
for t, df in RAW.items():
    dcols = [c for c in df.columns if DATE_COL_PAT.search(c)]
    main_dense = (pd.NaT, pd.NaT); best_n = -1; mins, maxs = [], []
    for c in dcols:
        s = pd.to_datetime(df[c], errors='coerce'); lo, hi = robust_range(s)
        cov_rows.append({'table': t, 'column': c, 'n_valid': int(s.notna().sum()), 'date_min': s.min(), 'date_max': s.max(), 'dense_min': lo, 'dense_max': hi})
        mins.append(s.min()); maxs.append(s.max())
        if s.notna().sum() > best_n: best_n, main_dense = int(s.notna().sum()), (lo, hi)
    fmt = lambda x: '' if pd.isna(x) else str(pd.Timestamp(x).date())
    cat_rows.append({'table': t, 'var_name': f'es_{to_snake(t)}_raw', 'rows': len(df), 'cols': df.shape[1],
                     'file_MB': round((DIRS['raw'] / f'{t}.csv').stat().st_size / 1e6, 2), 'mem_MB': round(df.memory_usage(deep=True).sum() / 1e6, 1),
                     'date_cols': ', '.join(dcols), 'date_min': fmt(min(mins) if mins else pd.NaT), 'date_max': fmt(max(maxs) if maxs else pd.NaT),
                     'dense_min': fmt(main_dense[0]), 'dense_max': fmt(main_dense[1])})
es_date_coverage = pd.DataFrame(cov_rows)
es_table_catalog = pd.DataFrame(cat_rows)
display(es_table_catalog)
print(f"Tổng: {es_table_catalog.rows.sum():,} dòng | {es_table_catalog.file_MB.sum():.1f} MB trên đĩa")

In [ ]:
# ---- Biểu đồ: thời gian phủ của từng bảng (thanh ngang). Xám nhạt = min–max thô; xanh = khoảng "dày đặc" (bỏ ngày lẻ tẻ) ----
cv = es_date_coverage.copy(); cv['label'] = cv['table'] + '.' + cv['column']; cv = cv.sort_values(['table', 'column']).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(13, 0.34 * len(cv) + 1.6))
for i, r in cv.iterrows():
    ax.barh(i, (r.date_max - r.date_min).days + 1, left=mdates.date2num(r.date_min), color='#d0d0d0', height=0.7, label='min–max thô' if i == 0 else None)
    ax.barh(i, (r.dense_max - r.dense_min).days + 1, left=mdates.date2num(r.dense_min), color='#4C78A8', height=0.4, label='khoảng dày đặc' if i == 0 else None)
ax.axvline(mdates.date2num(pd.Timestamp(SYNC_START)), c='red', ls='--', lw=1.2, label=f'SYNC_START {SYNC_START}')
ax.axvline(mdates.date2num(pd.Timestamp(SYNC_END) + pd.Timedelta(days=6)), c='darkorange', ls='--', lw=1.2, label=f'hết tuần SYNC_END')
ax.set_yticks(range(len(cv))); ax.set_yticklabels(cv['label'], fontsize=8); ax.invert_yaxis(); ax.xaxis_date()
ax.set_title('Thời gian phủ của từng bảng (cột ngày)'); ax.legend(loc='lower left', fontsize=8, ncol=2); plt.tight_layout(); plt.show(); save_fig(fig, 'thoi_gian_phu_cac_bang')

### 1.3 Chất lượng dữ liệu theo 6 chiều (bảng chính: `SalesOrders`, `SalesOrderLines`) → xuất `es_quality_sales`
Sáu chiều: **Đầy đủ · Duy nhất · Hợp lệ · Nhất quán · Chính xác/ngoại lai · Kịp thời (phạm vi thời gian)**. Mỗi dòng của bảng = một phép kiểm tra, có số dòng bị gắn cờ và tỷ lệ.

In [ ]:
D_COMP, D_UNIQ, D_VALID, D_CONS, D_ACC, D_TIME = ('1. Đầy đủ', '2. Duy nhất', '3. Hợp lệ', '4. Nhất quán', '5. Chính xác / ngoại lai', '6. Kịp thời / phạm vi thời gian')
QUALITY = []
def add_q(dim, table, check, n, total, note=''):
    QUALITY.append({'dimension': dim, 'table': table, 'check': check, 'n_flagged': int(n), 'n_total': int(total),
                    'pct': round(100 * float(n) / max(int(total), 1), 3), 'note': note})

so = es_sales_orders_raw.copy(); sol = es_sales_order_lines_raw.copy()
so['OrderDate'] = pd.to_datetime(so['OrderDate'], errors='coerce')
so_u = so.drop_duplicates()                                       # bản không trùng hệt
lo_d, hi_d = robust_range(so_u['OrderDate'])
print('Khoảng ngày đặt hàng "dày đặc":', lo_d.date(), '→', hi_d.date(), '| min–max thô:', so.OrderDate.min().date(), '→', so.OrderDate.max().date())
print('OrderStatus:', so_u['OrderStatus'].value_counts().to_dict())

# 1. ĐẦY ĐỦ
_ch = so_u.assign(m=so_u.SalesLocationID.isna()).merge(es_sales_channels_raw[['SalesChannelID', 'ChannelName']], on='SalesChannelID').groupby('ChannelName')['m'].mean()
NOTE_LOC = 'tỷ lệ thiếu theo kênh: ' + ', '.join(f'{k} {v * 100:.0f}%' for k, v in _ch.items()) + ' (đơn online không gắn cửa hàng bán)'
NOTE_MISS = {'PromotionID': 'thiếu có chủ đích: dòng không áp dụng khuyến mãi', 'CustomerID': 'đơn không gắn khách hàng', 'SalesLocationID': NOTE_LOC,
             'UnitCostAtSale': 'thiếu giá vốn tại thời điểm bán (không dùng làm đặc trưng bắt buộc)'}
for tname, d_ in [('SalesOrders', so_u), ('SalesOrderLines', sol)]:
    for c in d_.columns: add_q(D_COMP, tname, f'thiếu {c}', d_[c].isna().sum(), len(d_), NOTE_MISS.get(c, ''))
# 2. DUY NHẤT
add_q(D_UNIQ, 'SalesOrders', 'dòng trùng hoàn toàn', so.duplicated().sum(), len(so), 'phải drop_duplicates() TRƯỚC khi nối, nếu không doanh số bị nhân đôi')
add_q(D_UNIQ, 'SalesOrders', 'SalesOrderID trùng (sau khi bỏ dòng trùng hệt)', so_u.duplicated('SalesOrderID').sum(), len(so_u))
add_q(D_UNIQ, 'SalesOrderLines', 'SalesOrderLineID trùng', sol.duplicated('SalesOrderLineID').sum(), len(sol))
add_q(D_UNIQ, 'SalesOrderLines', '(SalesOrderID, LineNumber) trùng', sol.duplicated(['SalesOrderID', 'LineNumber']).sum(), len(sol))
add_q(D_UNIQ, 'Products', 'ProductID trùng (dòng trùng hệt)', es_products_raw.duplicated('ProductID').sum(), len(es_products_raw), 'phải drop_duplicates() trước khi nối với ProductVariants (nếu không SKU bị nhân đôi)')
add_q(D_UNIQ, 'ProductVariants', 'SKU_ID trùng', es_product_variants_raw.duplicated('SKU_ID').sum(), len(es_product_variants_raw))
# 3. HỢP LỆ
add_q(D_VALID, 'SalesOrders', 'OrderStatus viết khác chuẩn (Complete / Canceled)', so_u['OrderStatus'].isin(['Complete', 'Canceled']).sum(), len(so_u), 'cùng nghĩa với Completed / Cancelled; is_valid_order xử lý cả hai')
add_q(D_VALID, 'SalesOrders', 'OrderStatus là đơn hủy (loại khỏi doanh số)', (~is_valid_order(so_u)).sum(), len(so_u))
add_q(D_VALID, 'SalesOrders', 'OrderDate không đọc được', so_u['OrderDate'].isna().sum(), len(so_u))
add_q(D_VALID, 'SalesOrderLines', 'Quantity <= 0', (sol.Quantity <= 0).sum(), len(sol))
add_q(D_VALID, 'SalesOrderLines', 'Quantity không nguyên', (sol.Quantity % 1 != 0).sum(), len(sol))
add_q(D_VALID, 'SalesOrderLines', 'RegularUnitPrice <= 0', (sol.RegularUnitPrice <= 0).sum(), len(sol))
add_q(D_VALID, 'SalesOrderLines', 'ActualSellingPrice <= 0', (sol.ActualSellingPrice <= 0).sum(), len(sol))
add_q(D_VALID, 'SalesOrderLines', 'ActualSellingPrice > RegularUnitPrice', (sol.ActualSellingPrice > sol.RegularUnitPrice + 1e-9).sum(), len(sol))
# 4. NHẤT QUÁN
add_q(D_CONS, 'SalesOrderLines', 'dòng không có đơn hàng tương ứng', (~sol.SalesOrderID.isin(so_u.SalesOrderID)).sum(), len(sol))
add_q(D_CONS, 'SalesOrders', 'đơn không có dòng hàng', (~so_u.SalesOrderID.isin(sol.SalesOrderID)).sum(), len(so_u))
add_q(D_CONS, 'SalesOrderLines', 'SKU không có trong ProductVariants', (~sol.SKU_ID.isin(es_product_variants_raw.SKU_ID)).sum(), len(sol))
add_q(D_CONS, 'SalesOrderLines', 'PromotionID không có trong Promotions', (sol.PromotionID.notna() & ~sol.PromotionID.isin(es_promotions_raw.PromotionID)).sum(), int(sol.PromotionID.notna().sum()))
is_promo = sol.PromotionID.notna(); has_disc = sol.ActualSellingPrice < sol.RegularUnitPrice - 1e-9
add_q(D_CONS, 'SalesOrderLines', 'có PromotionID nhưng không giảm giá', (is_promo & ~has_disc).sum(), int(is_promo.sum()))
add_q(D_CONS, 'SalesOrderLines', 'giảm giá nhưng không có PromotionID', (~is_promo & has_disc).sum(), int((~is_promo).sum()), 'giảm giá không gắn chương trình → discount_rate vẫn tính từ giá thực tế')
_p = sol[is_promo].merge(so_u[['SalesOrderID', 'OrderDate']], on='SalesOrderID').merge(es_promotions_raw, on='PromotionID')
_out = (_p.OrderDate < pd.to_datetime(_p.StartDate)) | (_p.OrderDate > pd.to_datetime(_p.EndDate))
add_q(D_CONS, 'SalesOrderLines', 'đơn ngoài thời hạn của chương trình khuyến mãi', _out.sum(), len(_p))
_sh = es_shipments_raw[['SalesOrderID', 'ShipmentDate']].merge(so_u[['SalesOrderID', 'OrderDate']], on='SalesOrderID')
add_q(D_CONS, 'Shipments', 'ShipmentDate trước OrderDate', (pd.to_datetime(_sh.ShipmentDate, errors='coerce') < _sh.OrderDate).sum(), len(_sh))
# 5. CHÍNH XÁC / NGOẠI LAI
add_q(D_ACC, 'SalesOrderLines', 'Quantity >= 3', (sol.Quantity >= 3).sum(), len(sol), f'Quantity: min={sol.Quantity.min()}, max={sol.Quantity.max()}, trung vị={sol.Quantity.median():.0f} → phân bố lệch phải nhưng số lượng tối đa nhỏ, không có ngoại lai lớn')
_ratio = sol.RegularUnitPrice / sol.groupby('SKU_ID')['RegularUnitPrice'].transform('median')
add_q(D_ACC, 'SalesOrderLines', 'giá niêm yết lệch >2× hoặc <0.5× trung vị SKU', ((_ratio > 2) | (_ratio < 0.5)).sum(), len(sol))
add_q(D_ACC, 'SalesOrderLines', 'giảm giá sâu > 50% so với giá niêm yết', ((1 - sol.ActualSellingPrice / sol.RegularUnitPrice) > 0.5).sum(), len(sol))
add_q(D_ACC, 'SalesOrderLines', 'bán dưới giá vốn (ActualSellingPrice < UnitCostAtSale)', (sol.ActualSellingPrice < sol.UnitCostAtSale).sum(), int(sol.UnitCostAtSale.notna().sum()))
# 6. KỊP THỜI / PHẠM VI THỜI GIAN
add_q(D_TIME, 'SalesOrders', f'OrderDate ngoài khoảng dày đặc [{lo_d.date()}, {hi_d.date()}]', ((so_u.OrderDate < lo_d) | (so_u.OrderDate > hi_d)).sum(), len(so_u), 'ngoại lai thời gian: bị loại ở bước lọc cửa sổ SYNC')
add_q(D_TIME, 'SalesOrders', f'OrderDate sau REF_DATE ({REF_DATE.date()}) – ngày ở tương lai', (so_u.OrderDate > REF_DATE).sum(), len(so_u))
add_q(D_TIME, 'SalesOrders', f'đơn nằm ngoài cửa sổ SYNC [{SYNC_START}, {(pd.Timestamp(SYNC_END) + pd.Timedelta(days=6)).date()}]',
      ((so_u.OrderDate < SYNC_START) | (so_u.OrderDate > pd.Timestamp(SYNC_END) + pd.Timedelta(days=6))).sum(), len(so_u), 'gồm 2 tuần thiếu ngày ở hai đầu (01–02/07/2023 và 29–31/12/2025)')

es_quality_sales_part1 = pd.DataFrame(QUALITY)
display(es_quality_sales_part1.drop(columns=['n_total']))
print('\nSalesLocationID thiếu theo kênh bán:')
display(so_u.assign(thieu_SalesLocationID=so_u.SalesLocationID.isna()).groupby('SalesChannelID')['thieu_SalesLocationID'].agg(['mean', 'size']).rename(columns={'mean': 'tỷ_lệ_thiếu', 'size': 'số_đơn'}).join(es_sales_channels_raw.set_index('SalesChannelID')['ChannelName']))

### 1.4 Phần trăm thiếu theo cột của bảng chính

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, (nm, d_) in zip(ax, [('SalesOrders', so_u), ('SalesOrderLines', sol)]):
    pm = (d_.isna().mean() * 100).sort_values()
    a.barh(pm.index, pm.values, color=['#E45756' if v > 5 else '#4C78A8' for v in pm.values])
    for i, v in enumerate(pm.values): a.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=8)
    a.set_title(f'% thiếu theo cột – {nm}'); a.set_xlabel('% thiếu'); a.set_xlim(0, max(10, pm.max() * 1.2))
plt.tight_layout(); plt.show(); save_fig(fig, 'ty_le_thieu_bang_chinh')

### 1.5 Riêng tư: `CustomerName`
`CustomerName` là **định danh trực tiếp** (họ tên). Nhóm **không dùng** cột này: kiểm kê rồi **bỏ ngay khỏi bộ nhớ** trước mọi bước xử lý/xuất file.

In [ ]:
_cu = es_customers_raw
privacy = pd.DataFrame([
    {'bảng': 'Customers', 'cột': 'CustomerName', 'mức_độ': 'Định danh trực tiếp (họ tên)', 'số_giá_trị_khác_nhau': _cu.CustomerName.nunique(), 'tỷ_lệ_có_giá_trị_%': round(100 * _cu.CustomerName.notna().mean(), 1), 'cách_xử_lý': 'KHÔNG dùng; drop khỏi dữ liệu trước khi xử lý/xuất'},
    {'bảng': 'Customers / SalesOrders', 'cột': 'CustomerID', 'mức_độ': 'Mã định danh (giả danh)', 'số_giá_trị_khác_nhau': _cu.CustomerID.nunique(), 'tỷ_lệ_có_giá_trị_%': round(100 * so_u.CustomerID.notna().mean(), 1), 'cách_xử_lý': 'Không đưa vào đặc trưng/mô hình (mô hình ở mức SKU–tuần)'},
    {'bảng': 'Customers', 'cột': 'HomeGovernorateID, AgeBand', 'mức_độ': 'Bán định danh (địa lý, nhóm tuổi)', 'số_giá_trị_khác_nhau': _cu.HomeGovernorateID.nunique(), 'tỷ_lệ_có_giá_trị_%': round(100 * _cu.HomeGovernorateID.notna().mean(), 1), 'cách_xử_lý': 'Không dùng; nếu dùng phải gộp nhóm'}])
display(privacy)
name_cols = [(t, c) for t, d_ in RAW.items() for c in d_.columns if 'Name' in c]
print('Các cột có "Name" trong 27 bảng:', name_cols, '\n→ chỉ CustomerName là tên người; còn lại là tên sản phẩm/nhãn hàng/nhà cung cấp/địa danh.')
es_customers_raw = es_customers_raw.drop(columns=['CustomerName']); RAW['Customers'] = es_customers_raw
assert 'CustomerName' not in es_customers_raw.columns
add_q('7. Riêng tư', 'Customers', 'CustomerName – định danh trực tiếp', len(_cu), len(_cu), 'đã bỏ khỏi bộ nhớ, không xuất ra file nào'); del _cu

### 1.6 Phân bố `Quantity` – **Hình 1**

In [ ]:
_st = so_u.drop_duplicates('SalesOrderID').set_index('SalesOrderID')['OrderStatus']
sol_s = sol.assign(valid=sol.SalesOrderID.map(_st).pipe(lambda s: ~s.isin(['Cancelled', 'Canceled'])))
vc = sol.Quantity.value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(vc.index.astype(str), vc.values, color='#4C78A8'); ax[0].set_yscale('log')
for i, (q, n) in enumerate(vc.items()): ax[0].text(i, n * 1.1, f'{n:,}\n({100 * n / vc.sum():.1f}%)', ha='center', fontsize=8)
ax[0].set_title('Hình 1a – Số dòng hàng theo Quantity (trục log)'); ax[0].set_xlabel('Quantity trên một dòng đơn'); ax[0].set_ylim(top=vc.max() * 8)
cmp = pd.crosstab(sol_s.Quantity, sol_s.valid, normalize='columns') * 100
cmp.plot.bar(ax=ax[1], color=['#E45756', '#54A24B'], rot=0); ax[1].legend(['đơn hủy', 'đơn hợp lệ']); ax[1].set_ylabel('% số dòng'); ax[1].set_title('Hình 1b – Phân bố Quantity: đơn hợp lệ vs đơn hủy')
plt.tight_layout(); plt.show(); save_fig(fig, 'hinh1_phan_bo_quantity')
print(sol.Quantity.describe().round(3).to_dict())

## 2. Tạo nhãn và đặc trưng doanh số *(A11)*
### 2.1 Lọc đơn hợp lệ → nối `SalesOrderLines` với `SalesOrders` để lấy ngày đơn
Mọi bước ghi vào `es_filter_log` (số dòng **trước/sau**).

In [ ]:
FILTER_LOG = []
def log_filter(step, before, after, note=''):
    FILTER_LOG.append({'bước': step, 'số_dòng_trước': int(before), 'số_dòng_sau': int(after), 'thay_đổi': int(after - before), 'ghi_chú': note})
    print(f'  [{step}] {int(before):>9,} → {int(after):>9,}  (thay đổi {int(after - before):+,})  {note}')

W_START, W_END = pd.Timestamp(SYNC_START), pd.Timestamp(SYNC_END) + pd.Timedelta(days=6)      # hết ngày chủ nhật của tuần SYNC_END
log_filter('0. SalesOrderLines gốc', len(sol), len(sol), 'bảng dòng hàng')
orders = so.drop_duplicates()
log_filter('1. SalesOrders: bỏ dòng trùng hệt', len(so), len(orders), 'đơn vị = đơn hàng')
assert not orders.duplicated('SalesOrderID').any(), 'SalesOrderID vẫn trùng sau drop_duplicates'
n_before = len(orders); orders_valid = orders[is_valid_order(orders)]
log_filter('2. SalesOrders: giữ đơn hợp lệ (is_valid_order)', n_before, len(orders_valid), 'loại đơn Cancelled/Canceled; đơn vị = đơn hàng')
lines = sol.merge(orders_valid[['SalesOrderID', 'OrderDate', 'OrderStatus']], on='SalesOrderID', how='inner')
log_filter('3. Nối SalesOrderLines ⋈ đơn hợp lệ (lấy OrderDate)', len(sol), len(lines), 'inner join: dòng của đơn hủy bị loại')
lines = lines.rename(columns={'SKU_ID': 'sku_id'})
lines_valid_all = lines.copy()                                    # giữ bản chưa cắt cửa sổ để vẽ Hình 3 theo tháng đủ ngày
n_before = len(lines); lines = lines[(lines.OrderDate >= W_START) & (lines.OrderDate <= W_END + pd.Timedelta(hours=23, minutes=59))]
log_filter(f'4. Cửa sổ SYNC [{SYNC_START}, {W_END.date()}]', n_before, len(lines), 'bỏ ngày lẻ ngoài khoảng + 2 tuần thiếu ngày ở hai đầu')
n_before = len(lines); lines = lines[(lines.Quantity > 0) & (lines.ActualSellingPrice > 0) & (lines.RegularUnitPrice > 0)]
log_filter('5. Bỏ dòng Quantity/giá không hợp lệ (<= 0)', n_before, len(lines), 'kỳ vọng 0 dòng (xem chiều Hợp lệ)')
lines['week_start'] = week_start_of(lines['OrderDate'])
print(f'\nDòng hợp lệ dùng để tổng hợp: {len(lines):,} | SKU: {lines.sku_id.nunique()} | tuần: {lines.week_start.nunique()} ({lines.week_start.min().date()} → {lines.week_start.max().date()})')

### 2.2 Hình 2 (Quantity theo tuần của 3 SKU bán cao nhất) và Hình 3 (lượng bán theo tháng)

In [ ]:
top3 = lines.groupby('sku_id')['Quantity'].sum().nlargest(3).index.tolist()
all_weeks = pd.date_range(SYNC_START, SYNC_END, freq='7D')
wk3 = lines[lines.sku_id.isin(top3)].groupby(['sku_id', 'week_start'])['Quantity'].sum().unstack('sku_id').reindex(all_weeks, fill_value=0).fillna(0)
fig, ax = plt.subplots(figsize=(13, 4))
for s in top3: ax.plot(wk3.index, wk3[s], lw=1.2, label=f'{s} (tổng {int(wk3[s].sum()):,})')
ax.set_title('Hình 2 – Quantity theo tuần của 3 SKU bán cao nhất'); ax.set_ylabel('Số lượng bán / tuần'); ax.legend(); plt.tight_layout(); plt.show(); save_fig(fig, 'hinh2_top3_sku_theo_tuan')

lm = lines_valid_all[(lines_valid_all.OrderDate >= '2023-07-01') & (lines_valid_all.OrderDate <= REF_DATE + pd.Timedelta(hours=23))].copy()
lm['month'] = lm.OrderDate.dt.to_period('M').dt.to_timestamp(); lm['rev'] = lm.Quantity * lm.ActualSellingPrice
mo = lm.groupby('month').agg(qty=('Quantity', 'sum'), rev=('rev', 'sum'))
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(mo.index, mo.qty, width=20, color='#4C78A8', label='Số lượng bán'); ax.set_ylabel('Số lượng bán / tháng')
ax2 = ax.twinx(); ax2.plot(mo.index, mo.rev / 1e6, c='#F58518', marker='o', ms=3, label='Doanh thu (triệu)'); ax2.set_ylabel('Doanh thu (triệu, đơn vị tiền tệ gốc)'); ax2.grid(False)
ax.set_title('Hình 3 – Lượng bán theo tháng (đơn hợp lệ, 07/2023 → 12/2025)'); h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc='upper left')
plt.tight_layout(); plt.show(); save_fig(fig, 'hinh3_luong_ban_theo_thang')
display(mo.assign(qty_vs_thang_truoc_pct=(mo.qty.pct_change() * 100).round(1)).tail(8))

### 2.3 Tổng hợp SKU–tuần: `qty`, `revenue`, `avg_price`, `discount_rate`, `promo_flag`
* `qty` = tổng Quantity; `revenue` = Σ Quantity × ActualSellingPrice; `avg_price` = revenue ÷ qty (tuần không bán → giữ giá tuần gần nhất, **chỉ nhìn về quá khứ**).
* `discount_rate` = 1 − Σ(Q×ActualSellingPrice) ÷ Σ(Q×RegularUnitPrice) (có trọng số số lượng); `promo_flag` = 1 nếu có ít nhất một dòng gắn `PromotionID`.
* Điền `qty = 0` cho các tuần không phát sinh đơn **trong khoảng từ tuần bán đầu tiên đến tuần bán cuối cùng của từng SKU**.

In [ ]:
lines['rev_line'] = lines.Quantity * lines.ActualSellingPrice; lines['reg_line'] = lines.Quantity * lines.RegularUnitPrice
lines['promo_q'] = lines.Quantity.where(lines.PromotionID.notna(), 0)
agg = lines.groupby(KEY, as_index=False).agg(qty=('Quantity', 'sum'), revenue=('rev_line', 'sum'), reg_revenue=('reg_line', 'sum'), promo_qty=('promo_q', 'sum'))
log_filter('6. Tổng hợp dòng → SKU–tuần', len(lines), len(agg), 'gộp dòng hàng thành một dòng cho mỗi cặp (sku_id, week_start) có bán')

rng_ = agg.groupby('sku_id')['week_start'].agg(['min', 'max'])
grid = pd.concat([pd.DataFrame({'sku_id': s, 'week_start': pd.date_range(r['min'], r['max'], freq='7D')}) for s, r in rng_.iterrows()], ignore_index=True)
grid['week_start'] = grid['week_start'].astype('datetime64[ns]')
es_sales_wk = grid.merge(agg, on=KEY, how='left')
log_filter('7. Điền qty = 0 trong khoảng đầu–cuối của từng SKU', len(agg), len(es_sales_wk), f'thêm {len(es_sales_wk) - len(agg):,} dòng tuần-không-bán')
for c in ['qty', 'revenue', 'reg_revenue', 'promo_qty']: es_sales_wk[c] = es_sales_wk[c].fillna(0)
es_sales_wk['qty'] = es_sales_wk['qty'].astype(int)
es_sales_wk = es_sales_wk.sort_values(KEY).reset_index(drop=True)
es_sales_wk['avg_price'] = (es_sales_wk.revenue / es_sales_wk.qty.where(es_sales_wk.qty > 0)).groupby(es_sales_wk.sku_id).ffill()      # ffill: chỉ dùng quá khứ
es_sales_wk['discount_rate'] = np.where(es_sales_wk.reg_revenue > 0, 1 - es_sales_wk.revenue / es_sales_wk.reg_revenue.where(es_sales_wk.reg_revenue > 0), 0.0)
es_sales_wk['promo_flag'] = (es_sales_wk.promo_qty > 0).astype(int)
print('Tổng qty sau tổng hợp:', f'{es_sales_wk.qty.sum():,}', '| tổng Quantity dòng hợp lệ trong cửa sổ:', f'{lines.Quantity.sum():,}', '→', 'KHỚP ✅' if es_sales_wk.qty.sum() == lines.Quantity.sum() else 'LỆCH ❌')
assert es_sales_wk.qty.sum() == lines.Quantity.sum()
print('Số SKU:', es_sales_wk.sku_id.nunique(), '| số dòng SKU–tuần:', f'{len(es_sales_wk):,}', '| tỷ lệ tuần qty = 0:', round((es_sales_wk.qty == 0).mean(), 4))

### 2.4 Đặc trưng trễ/cuộn (chỉ dùng tuần ≤ t) và nhãn `y_qty`
* `qty_lag1/2/4/8` · `qty_rmean4/8`, `qty_rstd4/8` (cửa sổ gồm tuần t và các tuần trước, cần đủ cửa sổ) · `qty_slope4` = hệ số góc hồi quy tuyến tính của `qty` trên 4 tuần t−3…t.
* **Nhãn** `y_qty = qty` của **tuần t+1**; dòng cuối của mỗi SKU không có nhãn (NaN). Tùy chọn: `y_qty_sum4` = tổng qty tuần t+1…t+4 (bật bằng cờ).

In [ ]:
ADD_MULTI_HORIZON = False        # True: thêm y_qty_sum4 (tổng qty t+1…t+4) – tùy chọn của task
g = es_sales_wk.groupby('sku_id', sort=False)['qty']
for l in (1, 2, 4, 8): es_sales_wk[f'qty_lag{l}'] = g.shift(l)
for w in (4, 8):
    es_sales_wk[f'qty_rmean{w}'] = g.transform(lambda x: x.rolling(w, min_periods=w).mean())
    es_sales_wk[f'qty_rstd{w}']  = g.transform(lambda x: x.rolling(w, min_periods=w).std())
_l3 = g.shift(3)
es_sales_wk['qty_slope4'] = (1.5 * (es_sales_wk.qty - _l3) + 0.5 * (es_sales_wk.qty_lag1 - es_sales_wk.qty_lag2)) / 5     # OLS slope trên x = 0..3
es_sales_wk['y_qty'] = g.shift(-1)                                                                                          # NHÃN: qty tuần t+1
if ADD_MULTI_HORIZON:
    es_sales_wk['y_qty_sum4'] = sum(g.shift(-k) for k in range(1, HORIZON_WEEKS + 1))

# --- kiểm tra chống rò rỉ: đặc trưng chỉ phụ thuộc tuần <= t ---
_chk = es_sales_wk[es_sales_wk.sku_id == top3[0]].reset_index(drop=True)
i0 = 40; assert _chk.qty_lag1[i0] == _chk.qty[i0 - 1] and _chk.qty_lag8[i0] == _chk.qty[i0 - 8] and _chk.y_qty[i0] == _chk.qty[i0 + 1]
assert abs(_chk.qty_rmean4[i0] - _chk.qty[i0 - 3:i0 + 1].mean()) < 1e-9
assert abs(_chk.qty_slope4[i0] - np.polyfit(np.arange(4), _chk.qty[i0 - 3:i0 + 1].values, 1)[0]) < 1e-9
last_rows = es_sales_wk.groupby('sku_id').tail(1)
assert last_rows.y_qty.isna().all() and es_sales_wk.groupby('sku_id').y_qty.apply(lambda s: s.isna().sum()).eq(1).all()
print('✅ Kiểm tra: lag/rolling/slope chỉ dùng tuần ≤ t; y_qty = qty tuần t+1; đúng 1 dòng cuối/SKU không có nhãn')

### 2.5 Thuộc tính sản phẩm (danh mục, nhãn hàng, tuổi sản phẩm) – tất cả đã biết tại thời điểm t

In [ ]:
prod = (es_product_variants_raw[['SKU_ID', 'ProductID']]
        .merge(es_products_raw.drop_duplicates()[['ProductID', 'BrandID', 'SubcategoryID', 'ActiveFrom', 'ActiveTo']], on='ProductID', how='left')
        .merge(es_subcategories_raw, on='SubcategoryID', how='left').merge(es_categories_raw, on='CategoryID', how='left').merge(es_brands_raw, on='BrandID', how='left')
        .rename(columns={'SKU_ID': 'sku_id', 'CategoryName': 'category', 'SubcategoryName': 'subcategory', 'BrandName': 'brand'}))
assert prod.sku_id.is_unique
prod['ActiveFrom'] = pd.to_datetime(prod['ActiveFrom'])
es_sales_wk = es_sales_wk.merge(prod[['sku_id', 'category', 'subcategory', 'brand', 'ActiveFrom']], on='sku_id', how='left')
es_sales_wk['product_age_weeks'] = ((es_sales_wk.week_start - es_sales_wk.ActiveFrom).dt.days // 7).clip(lower=0)
es_sales_wk = es_sales_wk.drop(columns=['ActiveFrom', 'reg_revenue', 'promo_qty'])
FIRST = KEY + ['qty', 'y_qty', 'revenue', 'avg_price', 'discount_rate', 'promo_flag']
LAGS = [f'qty_lag{l}' for l in (1, 2, 4, 8)] + [f'qty_rmean{w}' for w in (4, 8)] + [f'qty_rstd{w}' for w in (4, 8)] + ['qty_slope4']
es_sales_wk = es_sales_wk[FIRST + LAGS + (['y_qty_sum4'] if ADD_MULTI_HORIZON else []) + ['category', 'subcategory', 'brand', 'product_age_weeks']]
print('Thiếu category/brand:', int(es_sales_wk.category.isna().sum()), int(es_sales_wk.brand.isna().sum()))
display(es_sales_wk.head(3)); display(es_sales_wk.isna().mean().round(4).to_frame('tỷ_lệ_thiếu').T)
n_short = int((es_sales_wk.groupby('sku_id').size() < MIN_HISTORY_WEEKS).sum())
print(f'Thông tin: {n_short} SKU có < {MIN_HISTORY_WEEKS} tuần quan sát toàn kỳ (bộ lọc MIN_HISTORY_WEEKS chính thức áp dụng trên TRAIN ở notebook chung)')

### 2.6 Thống kê phân bố của `y_qty` và biểu đồ

In [ ]:
def describe_target(s):
    s = s.dropna(); q = s.quantile([.25, .5, .75, .9, .95, .99])
    return pd.Series({'n': len(s), 'mean': s.mean(), 'median': s.median(), 'std': s.std(), 'min': s.min(), 'P25': q[.25], 'P75': q[.75], 'P90': q[.9], 'P95': q[.95], 'P99': q[.99], 'max': s.max(), 'tỷ_lệ_bằng_0': (s == 0).mean()})
tbl_y = describe_target(es_sales_wk.y_qty).round(3).to_frame('y_qty (toàn bộ)')
display(tbl_y)
tbl_y_cat = es_sales_wk.groupby('category')['y_qty'].apply(describe_target).unstack().round(3).sort_values('mean', ascending=False)
display(tbl_y_cat)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
y = es_sales_wk.y_qty.dropna(); ax[0].hist(y, bins=np.arange(0, y.quantile(0.999) + 2) - 0.5, color='#4C78A8'); ax[0].set_yscale('log')
ax[0].set_title('Histogram y_qty (trục log, cắt P99.9)'); ax[0].set_xlabel('y_qty (qty tuần t+1)'); ax[0].set_ylabel('số mẫu')
nw = es_sales_wk.groupby('sku_id').size(); ax[1].hist(nw, bins=30, color='#54A24B'); ax[1].axvline(MIN_HISTORY_WEEKS, c='red', ls='--', label=f'MIN_HISTORY_WEEKS={MIN_HISTORY_WEEKS}')
ax[1].set_title(f'Số tuần quan sát mỗi SKU (n={len(nw)} SKU)'); ax[1].set_xlabel('số tuần'); ax[1].legend()
z = es_sales_wk.groupby('sku_id')['qty'].apply(lambda s: (s == 0).mean()); ax[2].hist(z, bins=30, color='#E45756'); ax[2].set_title('Tỷ lệ tuần qty = 0 theo SKU'); ax[2].set_xlabel('tỷ lệ tuần qty = 0')
plt.tight_layout(); plt.show(); save_fig(fig, 'phan_bo_nhan_y_qty')
zc = es_sales_wk.groupby('category')['qty'].apply(lambda s: (s == 0).mean()).sort_values()
fig, ax = plt.subplots(figsize=(9, 3.6)); ax.barh(zc.index, zc.values * 100, color='#E45756'); ax.set_xlabel('% tuần qty = 0'); ax.set_title('Tỷ lệ tuần qty = 0 theo nhóm sản phẩm (category)')
for i, v in enumerate(zc.values): ax.text(v * 100 + 0.5, i, f'{v * 100:.1f}%', va='center', fontsize=8)
plt.tight_layout(); plt.show(); save_fig(fig, 'ty_le_qty0_theo_nhom')

# ngoại lai ở mức SKU–tuần (Tukey 3×IQR theo từng SKU) -> ghi vào bảng chất lượng
_q = es_sales_wk.groupby('sku_id')['qty']; fence = _q.transform(lambda s: s.quantile(.75) + 3 * (s.quantile(.75) - s.quantile(.25)))
add_q(D_ACC, 'es_sales_wk', 'qty SKU–tuần > Q3 + 3·IQR của chính SKU đó', (es_sales_wk.qty > fence).sum(), len(es_sales_wk), f'qty max = {es_sales_wk.qty.max()}, P99.9 = {es_sales_wk.qty.quantile(.999):.0f}: ngoại lai nhẹ, KHÔNG xóa (là biến động doanh số thật)')
add_q(D_COMP, 'es_sales_wk', 'y_qty thiếu (dòng cuối mỗi SKU)', es_sales_wk.y_qty.isna().sum(), len(es_sales_wk), 'thiết kế: tuần cuối không có tuần t+1')

## 3. EDA tín hiệu thời gian *(A11 – Mục 5 / Experiment 1)*
### 3.1 Xu hướng doanh số theo tuần

In [ ]:
tot = es_sales_wk.groupby('week_start').agg(qty=('qty', 'sum'), revenue=('revenue', 'sum'))
t_ = np.arange(len(tot)); b_q = np.polyfit(t_, tot.qty.values, 1); b_r = np.polyfit(t_, tot.revenue.values, 1)
tot['trend_qty'] = np.polyval(b_q, t_); tot['ma4'] = tot.qty.rolling(4).mean()
slope_pct = 100 * b_q[0] / tot.qty.mean()
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(tot.index, tot.qty, lw=1, alpha=.6, label='tổng qty / tuần'); ax[0].plot(tot.index, tot.ma4, lw=2, label='trung bình trượt 4 tuần'); ax[0].plot(tot.index, tot.trend_qty, 'r--', label=f'xu hướng tuyến tính ({slope_pct:+.2f}%/tuần)')
ax[0].set_title('Xu hướng doanh số theo tuần – số lượng'); ax[0].legend(fontsize=8)
ax[1].plot(tot.index, tot.revenue / 1e6, c='#F58518'); ax[1].plot(tot.index, np.polyval(b_r, t_) / 1e6, 'r--'); ax[1].set_title('Doanh thu theo tuần (triệu)')
plt.tight_layout(); plt.show(); save_fig(fig, 'eda_xu_huong_tuan')
print(f'Tổng qty/tuần: TB={tot.qty.mean():.0f}, min={tot.qty.min()}, max={tot.qty.max()} | xu hướng: {b_q[0]:+.1f} đơn vị/tuần ({slope_pct:+.2f}% TB/tuần, ≈ {slope_pct * 52:+.1f}%/năm)')

### 3.2 Mùa vụ theo tuần trong năm và theo tháng *(chỉ số mùa vụ = qty ÷ đường xu hướng tuyến tính)*

In [ ]:
tot['season_idx'] = tot.qty / tot.trend_qty
tot['woy'] = tot.index.isocalendar().week.astype(int).values; tot['month'] = tot.index.month; tot['year'] = tot.index.year
woy = tot.groupby('woy')['season_idx'].agg(['mean', 'min', 'max', 'count']); mon = tot.groupby('month')['season_idx'].mean()
mon_year = tot.pivot_table(index='month', columns='year', values='season_idx')
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(woy.index, woy['mean'], c='#4C78A8', lw=1.5); ax[0].fill_between(woy.index, woy['min'], woy['max'], alpha=.2); ax[0].axhline(1, c='gray', ls=':')
ax[0].set_title('Chỉ số mùa vụ theo tuần trong năm (trung bình, dải min–max giữa các năm)'); ax[0].set_xlabel('tuần ISO trong năm')
mon_year.plot(ax=ax[1], marker='o', ms=3, lw=1, alpha=.7); ax[1].plot(mon.index, mon.values, 'k-', lw=2.5, label='trung bình'); ax[1].axhline(1, c='gray', ls=':'); ax[1].set_xticks(range(1, 13))
ax[1].set_title('Chỉ số mùa vụ theo tháng (mỗi đường là một năm)'); ax[1].set_xlabel('tháng'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show(); save_fig(fig, 'eda_mua_vu')
print('Tháng cao nhất:', int(mon.idxmax()), f'(chỉ số {mon.max():.2f}) | thấp nhất:', int(mon.idxmin()), f'(chỉ số {mon.min():.2f}) | biên độ max/min = {mon.max() / mon.min():.2f}×')
print('Nhận xét: 2023 chỉ có nửa năm sau (07–12) nên mỗi tháng chỉ có 2–3 năm quan sát → mùa vụ mang tính gợi ý.')

### 3.3 Autocorrelation của `qty` ở lag 1/2/4/8 (tổng toàn chuỗi và từng SKU)

In [ ]:
wide = es_sales_wk.pivot(index='sku_id', columns='week_start', values='qty')
LAGS_ACF = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
acf_sku = pd.DataFrame({l: wide.apply(lambda r, l=l: r.autocorr(l), axis=1) for l in LAGS_ACF})
acf_tot = pd.Series({l: tot.qty.autocorr(l) for l in LAGS_ACF})
tbl_acf = pd.DataFrame({'ACF tổng chuỗi': acf_tot, 'ACF SKU – trung vị': acf_sku.median(), 'ACF SKU – P25': acf_sku.quantile(.25), 'ACF SKU – P75': acf_sku.quantile(.75),
                        '% SKU có ACF > 0.2': (acf_sku > 0.2).mean() * 100}).loc[[1, 2, 4, 8]].round(3)
display(tbl_acf)
fig, ax = plt.subplots(figsize=(10, 4)); med = acf_sku.median()
ax.bar(LAGS_ACF, med, color='#4C78A8', label='trung vị ACF theo SKU'); ax.errorbar(LAGS_ACF, med, yerr=[med - acf_sku.quantile(.25), acf_sku.quantile(.75) - med], fmt='none', c='k', capsize=3)
ax.plot(LAGS_ACF, acf_tot, 'o-', c='#E45756', label='ACF tổng chuỗi'); ax.axhline(0, c='k', lw=.8)
for l in (1, 2, 4, 8): ax.axvline(l, c='gray', ls=':', lw=.7)
ax.set_xticks(LAGS_ACF); ax.set_xlabel('lag (tuần)'); ax.set_title('Autocorrelation của qty (thanh sai số = P25–P75 giữa các SKU)'); ax.legend(); plt.tight_layout(); plt.show(); save_fig(fig, 'eda_autocorrelation')

### 3.4 Quan hệ tồn kho – tốc độ bán (scatter `stock_end` với trung bình bán 4 tuần)
`stock_end` được tái lập từ `InventoryMovements` bằng đúng hàm `stock_end_chain_weekly` dùng chung (chi tiết kiểm chứng ở NB2).

In [ ]:
stk = stock_end_chain_weekly(es_inventory_movements_raw)
se = es_sales_wk.merge(stk[KEY + ['stock_end']], on=KEY, how='inner').dropna(subset=['qty_rmean4'])
print(f'Số mẫu SKU–tuần có cả stock_end và qty_rmean4: {len(se):,}')
rho_all = se['stock_end'].corr(se['qty_rmean4'], method='spearman')
rho_sku = se.groupby('sku_id').apply(lambda d: d['stock_end'].corr(d['qty_rmean4'], method='spearman')).dropna()
smp = se.sample(min(20000, len(se)), random_state=SEED)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].scatter(smp.qty_rmean4, smp.stock_end, s=4, alpha=.15, c='#4C78A8'); ax[0].set_xlabel('qty_rmean4 (TB bán 4 tuần gần nhất)'); ax[0].set_ylabel('stock_end'); ax[0].set_title(f'stock_end vs tốc độ bán (mẫu {len(smp):,}) – Spearman = {rho_all:.2f}')
bins = pd.qcut(se.qty_rmean4, 10, duplicates='drop'); bm = se.groupby(bins, observed=True)['stock_end'].median()
ax[1].plot(range(len(bm)), bm.values, 'o-', c='#E45756'); ax[1].set_xticks(range(len(bm))); ax[1].set_xticklabels([f'{i.mid:.1f}' for i in bm.index], rotation=45); ax[1].set_xlabel('nhóm qty_rmean4 (điểm giữa, theo decile)'); ax[1].set_ylabel('stock_end – trung vị'); ax[1].set_title('Trung vị stock_end theo decile tốc độ bán')
plt.tight_layout(); plt.show(); save_fig(fig, 'eda_ton_kho_vs_toc_do_ban')
print(f'Spearman(stock_end, qty_rmean4): toàn bộ = {rho_all:.3f} | trung vị trong từng SKU = {rho_sku.median():.3f} (P25 {rho_sku.quantile(.25):.2f}, P75 {rho_sku.quantile(.75):.2f})')

### 3.5 Kết luận: dữ liệu có đủ tín hiệu để dự báo không?
Kiểm tra định lượng nhanh (không phải mô hình chính): trên **20% tuần cuối** (cách train 4 tuần), so sánh các dự báo đơn giản với mức trung bình của SKU lấy từ train. Nếu dự báo dựa trên lịch sử gần đây tốt hơn rõ rệt → có tín hiệu tự hồi quy để mô hình khai thác.

In [ ]:
d_ = es_sales_wk.dropna(subset=['y_qty', 'qty_rmean4', 'qty_rmean8']); wk_all = np.sort(d_.week_start.unique()); n_w = len(wk_all)
tr_end = wk_all[int(0.8 * n_w) - 1]; te_start = wk_all[min(int(0.8 * n_w) - 1 + HORIZON_WEEKS, n_w - 1)]
tr, te = d_[d_.week_start <= tr_end], d_[d_.week_start >= te_start]
wape = lambda y, p: float(np.abs(y - p).sum() / max(np.abs(y).sum(), EPS))
sku_mean = tr.groupby('sku_id')['qty'].mean(); base = te.sku_id.map(sku_mean).fillna(tr.qty.mean())
res_sig = pd.DataFrame({'WAPE': {'TB train của SKU (không dùng lịch sử gần)': wape(te.y_qty, base), 'Naive (qty tuần t)': wape(te.y_qty, te.qty),
                                  'TB trượt 4 tuần': wape(te.y_qty, te.qty_rmean4), 'TB trượt 8 tuần': wape(te.y_qty, te.qty_rmean8)}})
res_sig['MAE'] = [float(np.abs(te.y_qty - p).mean()) for p in (base, te.qty, te.qty_rmean4, te.qty_rmean8)]
print(f'Tập thử: {len(te):,} mẫu, tuần {pd.Timestamp(te_start).date()} → {pd.Timestamp(wk_all[-1]).date()}')
res_sig.loc['TB toàn cục của mọi SKU (không phân biệt SKU)'] = [wape(te.y_qty, np.full(len(te), tr.qty.mean())), float(np.abs(te.y_qty - tr.qty.mean()).mean())]
display(res_sig.round(4))
w_glob, w_sku = res_sig.loc['TB toàn cục của mọi SKU (không phân biệt SKU)', 'WAPE'], res_sig.iloc[0]['WAPE']
w_hist = res_sig.loc[['TB trượt 4 tuần', 'TB trượt 8 tuần'], 'WAPE'].min(); best_hist = res_sig.loc[['TB trượt 4 tuần', 'TB trượt 8 tuần'], 'WAPE'].idxmin()
gain_level, gain_hist = 1 - w_sku / w_glob, 1 - w_hist / w_sku
a = tbl_acf['ACF SKU – trung vị']; a1, a2, a4, a8 = a.loc[1], a.loc[2], a.loc[4], a.loc[8]; tot_a1 = tbl_acf.loc[1, 'ACF tổng chuỗi']
sig_level = gain_level >= 0.10                                  # biết SKU nào bán nhanh/chậm đã giảm sai số ≥ 10%
sig_hist  = (gain_hist >= 0.05) or (a1 >= 0.20)                 # lịch sử gần cải thiện ≥ 5% so với trung bình SKU, hoặc ACF lag1 ≥ 0.2
sig_agg   = (tot_a1 >= 0.5) and (mon.max() / mon.min() >= 1.10) # tổng chuỗi có tự tương quan mạnh và mùa vụ ≥ 10%
print('================ KẾT LUẬN TỰ ĐỘNG (số liệu chạy lại được; ngưỡng ghi rõ trong code) ================')
print(f'• Xu hướng: {slope_pct:+.2f}% TB/tuần (≈ {slope_pct * 52:+.1f}%/năm).')
print(f'• Mùa vụ: biên độ tháng cao/thấp = {mon.max() / mon.min():.2f}× (cao nhất tháng {int(mon.idxmax())}, thấp nhất tháng {int(mon.idxmin())}).')
print(f'• ACF theo SKU (trung vị): lag1={a1:.2f}, lag2={a2:.2f}, lag4={a4:.2f}, lag8={a8:.2f} | ACF tổng chuỗi lag1={tot_a1:.2f}.')
print(f'• Tín hiệu MỨC (SKU nào bán nhanh/chậm): biết trung bình SKU giảm WAPE {gain_level * 100:.1f}% so với trung bình toàn cục → {"CÓ" if sig_level else "YẾU"}.')
print(f'• Tín hiệu ĐỘNG theo thời gian ở mức SKU: {best_hist} giảm WAPE {gain_hist * 100:.1f}% so với trung bình SKU → {"CÓ" if sig_hist else "YẾU"}.')
print(f'• Tín hiệu ở mức TỔNG (xu hướng + mùa vụ + tự tương quan): {"CÓ" if sig_agg else "YẾU"}.')
print(f'• Tồn kho – tốc độ bán: Spearman toàn bộ = {rho_all:.2f}, trung vị trong từng SKU = {rho_sku.median():.2f} ' + ('(giữa các SKU: bán nhanh thì tồn nhiều; trong một SKU: tồn tăng khi bán chậm lại).' if rho_all > 0 > rho_sku.median() else '.'))
enough = sig_level and (sig_hist or sig_agg)
print('→ KẾT LUẬN:', 'DỮ LIỆU ĐỦ TÍN HIỆU để dự báo qty tuần t+1: có tín hiệu về mức của từng SKU và ' + ('tín hiệu động từ lịch sử gần. ' if sig_hist else 'tín hiệu xu hướng/mùa vụ ở mức tổng; tín hiệu động riêng từng SKU còn yếu (chuỗi SKU nhiều nhiễu) nên kỳ vọng mô hình chỉ cải thiện vừa phải so với baseline TB trượt – cần thêm đặc trưng mùa vụ/giá/khuyến mãi.') if enough
      else 'TÍN HIỆU YẾU: cần thận trọng với kỳ vọng về mô hình.')
print('• Hạn chế: chuỗi ngắn (≈130 tuần), mùa vụ chỉ quan sát 2–3 chu kỳ, qty từng SKU–tuần nhỏ (TB ≈ 8) nên nhiễu ngẫu nhiên lớn.')

## 4. Kiểm tra khóa → ghi file xuất
Với từng file: **khóa không trùng → số dòng → tuần min–max → số thiếu → mới ghi**.

In [ ]:
# ---- hợp đồng file (Mục 2.3 notebook chung) ----
REQ = ['sku_id', 'week_start', 'qty', 'y_qty']; assert all(c in es_sales_wk.columns for c in REQ)
assert not set(es_sales_wk.columns) & {'risk', 'risk_name', 'cov_fwd', 'CustomerName'}
assert es_sales_wk.sku_id.nunique() == lines.sku_id.nunique(), 'phải xuất TOÀN BỘ SKU hợp lệ, không chỉ một phần'
es_quality_sales = pd.DataFrame(QUALITY); es_filter_log = pd.DataFrame(FILTER_LOG)
print(f'Xuất cho TOÀN BỘ {es_sales_wk.sku_id.nunique()} SKU hợp lệ (không chỉ 20 SKU)\n')
check_and_save(es_sales_wk,      'es_sales_wk',      key=KEY)
check_and_save(es_filter_log,    'es_filter_log')
check_and_save(es_quality_sales, 'es_quality_sales')
check_and_save(es_table_catalog, 'es_table_catalog')
print('File trong interim/:', sorted(p.name for p in DIRS['interim'].iterdir()))
display(es_filter_log)